### 法律数据集清洗、去重、标准化

In [ ]:
# 法律数据集 TXT 版：清洗 + 去重 + 标准化
import re

# 1. 读取原始法律文本
with open("laws.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print("原始总行数：", len(lines))

# 2. 清洗每一行
cleaned = []
for line in lines:
    # 去掉换行、空格
    line = line.strip()
    
    # 跳过空行
    if not line:
        continue
    
    # 去掉乱码、控制字符
    line = re.sub(r'[\x00-\x1F\x7F]', '', line)
    
    # 只保留中文、数字、常用法律标点
    line = re.sub(r'[^\u4e00-\u9fa50-9，。！？；：、（）【】《》——]', '', line)
    
    # 再次去掉空行
    if len(line) < 5:  # 太短的无意义行删掉
        continue
    
    cleaned.append(line)

print("清洗后行数：", len(cleaned))

# 3. 去重（保持顺序）
unique_lines = []
seen = set()
for line in cleaned:
    if line not in seen:
        seen.add(line)
        unique_lines.append(line)

print("去重后最终行数：", len(unique_lines))

# 4. 标准化保存
with open("clean_laws.txt", "w", encoding="utf-8") as f:
    for line in unique_lines:
        f.write(line + "\n")

print("处理完成！已生成：clean_laws.txt")

原始总行数： 4859948
清洗后行数： 2519387
去重后最终行数： 1744419
✅ 处理完成！已生成：clean_laws.txt


### 法律意图分类、实体识别、关键词提取

In [ ]:
import pandas as pd
import jieba
import re
from collections import Counter

# 1. 加载清洗好的法律文本
with open("clean_laws.txt", "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f.readlines() if line.strip()]

print(f"加载文本数量：{len(texts)}")

# 2. 法律意图分类（基于关键词匹配，适合课程项目）
# 定义常见法律意图类别和对应关键词
intent_rules = {
    "合同纠纷": ["合同", "协议", "违约", "条款", "解除", "履行", "无效"],
    "劳动争议": ["工资", "劳动合同", "试用期", "辞退", "加班", "社保", "工伤"],
    "婚姻家庭": ["离婚", "财产分割", "抚养权", "赡养", "继承", "遗嘱"],
    "借贷纠纷": ["借条", "欠款", "利息", "还款", "债务", "担保"],
    "侵权责任": ["赔偿", "侵权", "人身损害", "名誉权", "肖像权", "交通事故"],
    "知识产权": ["专利", "商标", "著作权", "抄袭", "侵权", "盗版"],
    "其他": []
}

def classify_intent(text):
    """基于关键词匹配进行意图分类"""
    for intent, keywords in intent_rules.items():
        for kw in keywords:
            if kw in text:
                return intent
    return "其他"

# 3. 实体识别（提取法律相关实体）
def extract_entities(text):
    """提取文本中的法律实体（简单版）"""
    entities = {
        "法律文件": [],
        "主体": [],
        "行为": []
    }
    # 法律文件实体
    law_docs = ["合同", "协议", "借条", "遗嘱", "判决书", "调解书"]
    # 主体实体
    subjects = ["甲方", "乙方", "用人单位", "劳动者", "债权人", "债务人", "原告", "被告"]
    # 行为实体
    actions = ["解除", "履行", "违约", "赔偿", "辞退", "借款", "继承", "侵权"]

    for doc in law_docs:
        if doc in text:
            entities["法律文件"].append(doc)
    for subj in subjects:
        if subj in text:
            entities["主体"].append(subj)
    for act in actions:
        if act in text:
            entities["行为"].append(act)
    
    # 去重
    for k in entities:
        entities[k] = list(set(entities[k]))
    return entities


# 4. 关键词提取（TF-IDF简化版，基于词频）
def extract_keywords(text, top_k=5):
    """基于词频提取关键词"""
    # 分词
    words = jieba.lcut(text)
    # 过滤停用词和短词
    stopwords = {"的", "了", "和", "是", "在", "有", "对", "就", "也", "都", "等", "可以", "应当", "应当", "不得", "可以"}
    filtered_words = [w for w in words if len(w) > 1 and w not in stopwords]
    # 统计词频并取topK
    word_counts = Counter(filtered_words)
    keywords = [w for w, cnt in word_counts.most_common(top_k)]
    return keywords

# 5. 批量处理并保存结果
results = []
for text in texts:
    intent = classify_intent(text)
    entities = extract_entities(text)
    keywords = extract_keywords(text)
    results.append({
        "text": text,
        "intent": intent,
        "entities": entities,
        "keywords": keywords
    })

# 转为DataFrame并保存
df = pd.DataFrame(results)
df.to_csv("laws_structured.csv", index=False, encoding="utf-8-sig")

print("处理完成！结构化结果已保存为 laws_structured.csv")
print("\n前3条结果示例：")
print(df[["text", "intent", "entities", "keywords"]].head(3))

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\毛姐\AppData\Local\Temp\jieba.cache


加载文本数量：1744419


Loading model cost 0.598 seconds.
Prefix dict has been built successfully.


✅ 处理完成！结构化结果已保存为 laws_structured.csv

前3条结果示例：
                    text intent                          entities  \
0  标题中华人民共和国宪法修正案（2018年）     其他  {'法律文件': [], '主体': [], '行为': []}   
1                发布日期无日期     其他  {'法律文件': [], '主体': [], '行为': []}   
2         内容中华人民共和国宪法修正案     其他  {'法律文件': [], '主体': [], '行为': []}   

                     keywords  
0  [标题, 中华人民共和国宪法, 修正案, 2018]  
1                    [日期, 发布]  
2        [内容, 中华人民共和国宪法, 修正案]  


### 大模型 prompt 设计、优化、测试 
### +多轮对话 + 上下文记忆 + 对话管理核心逻辑

① 调用智谱 AI GLM-4-Flash代码  带多轮对话 + 上下文记忆 + 对话管理核心逻辑 的完整版GLM.py

In [21]:
from openai import OpenAI

# 智谱AI 免费配置
client = OpenAI(
    api_key="09ba18a1ad2042d0875c955a25bcbab5.8MSq01TiFZsR69MC",  # 注册后在控制台获取
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)

# 法律Prompt（优化版）
system_prompt = """
你是专业法律AI咨询助手，基于中国现行法律法规作答。
约束：
1. 仅依据中国大陆现行有效法律；
2. 不预测判决、不代写诉状、不提供诉讼代理；
3. 拒绝违法/灰色/侵权咨询；
4. 回答简洁通俗、条理清晰、分点说明；
5. 结合上下文连贯回答。
"""

# 多轮上下文
messages = [{"role": "system", "content": system_prompt}]

print("===== 智谱GLM-4-Flash｜法律对话 =====")
print("输入 exit 退出\n")

while True:
    user_q = input("你：")
    if user_q.lower() == "exit":
        print("AI：对话结束")
        break
    messages.append({"role": "user", "content": user_q})
    # 调用免费模型
    response = client.chat.completions.create(
        model="GLM-4-Flash",  # 永久免费
        messages=messages,
        temperature=0.3
    )
    ai_ans = response.choices[0].message.content
    print(f"AI：{ai_ans}\n")
    messages.append({"role": "assistant", "content": ai_ans})

===== 智谱GLM-4-Flash｜法律对话 =====
输入 exit 退出



AI：在中国，试用期被辞退是否有补偿，主要取决于以下几个因素：

1. **劳动合同法规定**：根据《中华人民共和国劳动合同法》的规定，如果用人单位在试用期内解除劳动合同，且没有法定理由（如劳动者不符合录用条件等），则应当向劳动者支付经济补偿。

2. **试用期限**：如果试用期内被辞退，补偿的数额通常与试用期的长短有关。试用期的长度根据劳动合同的约定和岗位性质而定。

3. **经济补偿标准**：经济补偿的标准通常是按劳动者在本单位工作的月工资为标准，每满一年支付一个月工资的经济补偿。六个月以上不满一年的，按一年计算；不满六个月的，向劳动者支付半个月工资的经济补偿。

4. **特殊情况**：如果劳动者在试用期内因患病或非因工负伤，在规定的医疗期满后不能从事原工作，也不能从事由用人单位另行安排的工作的，用人单位应当支付经济补偿。

总结来说，试用期被辞退是否有补偿，以及补偿的数额，需要根据具体情况和法律规定来确定。如果用人单位无正当理由在试用期内辞退劳动者，应当支付经济补偿。

AI：在试用期被辞退要求经济补偿时，以下几种证据是必要的：

1. **劳动合同**：这是证明双方存在劳动关系的基础文件，包括试用期条款。

2. **解除劳动合同的书面通知**：如果用人单位以书面形式通知劳动者解除劳动合同，该通知是证明解除事实的重要证据。

3. **工资支付凭证**：包括工资条、银行转账记录等，用以证明劳动者的月工资标准，这是计算经济补偿的依据。

4. **工作记录**：如考勤记录、工作总结、同事证言等，用以证明劳动者在试用期内的工作表现和出勤情况。

5. **医疗记录**（如有）：如果劳动者因健康原因被辞退，相关的医疗记录和病假证明可以作为证据。

6. **其他相关证据**：
   - **录音录像**：如果劳动者在解除劳动合同过程中有录音录像，且内容合法，可以作为证据。
   - **证人证言**：同事、上级或其他知情人提供的证言，证明劳动者的工作表现和被辞退的原因。
   - **公司规章制度**：如果公司有关于试用期的规章制度，且这些规定符合法律规定，可以作为证据。

需要注意的是，收集证据时应确保其合法性和有效性，避免使用非法手段获取证据。同时，证据应当尽可能全面，以支持劳动者的主张。在劳动争议中，这些证据将帮助劳动者证明自己的权益，并依法维护自己的

② 调用 Qwen 通义千问代码   带多轮对话 + 上下文记忆 + 对话管理核心逻辑 的完整版qwen.py

In [10]:
from openai import OpenAI

# 你的通义千问KEY
API_KEY = "sk-be42b2c30a0d4892a5c34a4b05754b4e"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# 系统提示词
system_prompt = """
你是专业法律咨询助手，遵守规则：
1. 依据中国现行法律回答
2. 不提供具体司法判决、不代书诉讼
3. 不提供违法建议
4. 回答简洁、准确、通俗
"""

# ========================
# 多轮对话上下文（核心）
# ========================
messages = [{"role": "system", "content": system_prompt}]

print("===== 法律AI聊天机器人（输入 exit 退出）=====\n")

while True:
    # 用户输入
    user_input = input("你：")

    # 退出
    if user_input.lower() == "exit":
        print("AI：再见！")
        break

    # 把用户问题加入上下文
    messages.append({"role": "user", "content": user_input})

    # 请求模型
    response = client.chat.completions.create(
        model="qwen-turbo",
        messages=messages,
        temperature=0.3,
        max_tokens=1024
    )

    # 获取回答
    answer = response.choices[0].message.content

    # 输出AI回答（干净聊天格式）
    print("AI：", answer, "\n")

    # 把AI回答也加入上下文
    messages.append({"role": "assistant", "content": answer})

===== 法律AI聊天机器人（输入 exit 退出）=====



AI： 根据中国现行法律，试用期被辞退是否需要补偿，需分情况讨论：

1. **用人单位解除劳动合同**：  
   若用人单位在试用期内以“不符合录用条件”为由解除劳动合同，**无需支付经济补偿**，但需提供证据证明劳动者不符合录用条件。

2. **劳动者主动辞职**：  
   若劳动者在试用期内主动辞职，**无权要求经济补偿**。

3. **违法解除**：  
   若用人单位在试用期内无合法理由解除劳动合同（如未提前通知、未说明理由等），劳动者可主张**违法解除赔偿金**，标准为经济补偿的2倍。

**总结**：  
- 用人单位合法解除试用期合同，**无需补偿**；  
- 劳动者主动辞职，**无补偿**；  
- 用人单位违法解除，**可要求赔偿**。  

建议保留相关证据，必要时咨询专业律师。 

AI： 在试用期被辞退的情况下，若你认为用人单位违法解除劳动合同，或要求经济补偿，需准备以下证据：

### 一、基本证据（通用）：
1. **劳动合同**：证明你与用人单位存在劳动关系。
2. **工资支付记录**：如银行流水、工资条等，证明你实际工作时间及工资情况。
3. **考勤记录**：证明你是否正常出勤、工作表现。

---

### 二、针对“不符合录用条件”的证据（用人单位解除时使用）：
1. **招聘时的岗位职责和录用条件**：如招聘信息、岗位说明书、入职培训材料等。
2. **考核记录**：如绩效评估、工作表现评价等，证明你确实不符合录用条件。
3. **通知解除的书面材料**：如解除通知书、邮件、短信等，证明用人单位已依法告知你解除理由。

---

### 三、针对“违法解除”的证据：
1. **用人单位解除通知**：如口头或书面通知，说明解除理由。
2. **无合法理由的证据**：如用人单位未提供符合录用条件的依据、未提前通知等。
3. **沟通记录**：如与用人单位负责人、HR的聊天记录、录音等，证明其解除行为不合法。

---

### 四、其他可能有用的证据：
- **工作群聊记录、邮件、工作成果等**：证明你在试用期内的工作表现。
- **证人证言**：如有同事可作证你的工作表现。

---

### 提示：
- 所有证据应尽量保留原件或可验证的电子记录；
- 若用人单位拒绝提供相关材料，可通过仲裁或诉讼程序申请法院调取